# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [1]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Flatten, TimeDistributed, Reshape
from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm

In [2]:
train_data = pd.read_csv('Dataset/train.csv')
test_data = pd.read_csv('Dataset/test.csv')

train_data.sample(5)

,parent_label,label,video_path,include_50
887,People,74. Queen,People/74. Queen/MVI_8641.MP4,False
2313,Colours,48. Green,Colours/48. Green/MVI_5188.mp4,False
2855,Society,18. Exercise,Society/18. Exercise/MVI_8980.MP4,False
1607,Jobs,96. Manager,Jobs/96. Manager/MVI_5365.mp4,False
3368,Jobs,91. Priest,Jobs/91. Priest/MVI_5344.mp4,True


In [3]:
# labels = train_data['label']
# train_data_path = train_data['vedio_path']


labels = []
train_path_I50 = []

for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        labels.append(train_data['label'][i])
        train_path_I50.append("Dataset\\" + train_data['video_path'][i])
        
labels = pd.Series(labels).unique()
labels = pd.Series(labels).to_list()

train_path_I50 = pd.Series(train_path_I50)

# train_path_I50.sample(5), labels.sample(5)
labels[:5]

['1. loud', '19. House', '83. big large', '91. Priest', '23. Court']

In [4]:
label_map = dict()

# for i in range(len(labels)):
    # split = labels[i].split(" ")
        # 
    # label = split[1:]
    # label = " ".join(label)
    # 
    # label_map[label] = int(split[0].split(".")[0])

# 
for i in range(len(labels)):
    # Splitting Label from num
    split = labels[i].split(" ")
    
    # Joining the ramining str to make a full label name
    label = split[1:]
    label = " ".join(label)
        
    label_map[label] = i

labels = list(label_map.keys())
  
label_map    

{'loud': 0,
 'House': 1,
 'big large': 2,
 'Priest': 3,
 'Court': 4,
 'train ticket': 5,
 'it': 6,
 'Shoes': 7,
 'Dog': 8,
 'Bank': 9,
 'Thank you': 10,
 'Election': 11,
 'Cow': 12,
 'Window': 13,
 'quiet': 14,
 'dry': 15,
 'long': 16,
 'Hello': 17,
 'Bird': 18,
 'Hat': 19,
 'Black': 20,
 'short': 21,
 'White': 22,
 'Fan': 23,
 'new': 24,
 'Store or Shop': 25,
 'Monday': 26,
 'Death': 27,
 'Cell phone': 28,
 'you (plural)': 29,
 'T-Shirt': 30,
 'Girl': 31,
 'Father': 32,
 'Red': 33,
 'hot': 34,
 'Fall': 35,
 'I': 36,
 'Time': 37,
 'Car': 38,
 'Good Morning': 39,
 'Summer': 40,
 'Paint': 41,
 'Teacher': 42,
 'Brother': 43,
 'good': 44,
 'happy': 45,
 'Boy': 46,
 'small little': 47,
 'Pen': 48,
 'Year': 49}

In [5]:
# Loading all the video frames
video_frames = []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    window = []
    
    for video in label_videos:
        # table = pq.read_table("MP_data/" + f"{label}/" + video)

        # df = table.to_pandas()

        # numpy_array = df.to_numpy()
        
        res = np.load("MP_data/" + f"{label}/" + video,allow_pickle=True)
        
        # window.append(numpy_array)
        window.append(res)
        
    video_frames.append(window)

print(len(video_frames))

        

  0%|          | 0/50 [00:00<?, ?it/s]

50


In [6]:
# Padding vedio sequences so all array are of equal length


# Checking whether the gpu is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        print("Using GPU:", gpus)

    except RuntimeError as e:
        print(e)

else:
    print("No GPU found.")




Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:


# Pad sequences to ensure they have the same length
max_length = max(len(seq) for seq in video_frames)
padded_video_frames = []

for frames in video_frames:
    padded_video_frames.append(pad_sequences(frames, maxlen=max_length, dtype='float32', padding='post', truncating='post'))

for i, frames in enumerate(padded_video_frames):
    print(f"Video {i+1} shape: {frames.shape}")

Video 1 shape: (13, 19, 1662)
Video 2 shape: (18, 19, 1662)
Video 3 shape: (15, 19, 1662)
Video 4 shape: (13, 19, 1662)
Video 5 shape: (15, 19, 1662)
Video 6 shape: (12, 19, 1662)
Video 7 shape: (11, 19, 1662)
Video 8 shape: (14, 19, 1662)
Video 9 shape: (12, 19, 1662)
Video 10 shape: (12, 19, 1662)
Video 11 shape: (13, 19, 1662)
Video 12 shape: (11, 19, 1662)
Video 13 shape: (16, 19, 1662)
Video 14 shape: (10, 19, 1662)
Video 15 shape: (16, 19, 1662)
Video 16 shape: (19, 19, 1662)
Video 17 shape: (14, 19, 1662)
Video 18 shape: (13, 19, 1662)
Video 19 shape: (17, 19, 1662)
Video 20 shape: (13, 19, 1662)
Video 21 shape: (14, 19, 1662)
Video 22 shape: (13, 19, 1662)
Video 23 shape: (17, 19, 1662)
Video 24 shape: (11, 19, 1662)
Video 25 shape: (16, 19, 1662)
Video 26 shape: (14, 19, 1662)
Video 27 shape: (11, 19, 1662)
Video 28 shape: (8, 19, 1662)
Video 29 shape: (9, 19, 1662)
Video 30 shape: (16, 19, 1662)
Video 31 shape: (15, 19, 1662)
Video 32 shape: (16, 19, 1662)
Video 33 shape: (16

In [8]:
print(padded_video_frames[0][0])

[[ 0.50861764  0.17247725 -0.60097396 ...  0.4135235   0.8987474
   0.00506143]
 [ 0.5098589   0.16221848 -0.5928083  ...  0.40933102  0.8993803
   0.00279614]
 [ 0.5104922   0.16071339 -0.58750767 ...  0.4110439   0.90037555
   0.00426812]
 ...
 [ 0.511447    0.1617201  -0.50619894 ...  0.44034308  0.3562057
   0.00172526]
 [ 0.5114116   0.16360673 -0.42434666 ...  0.4458513   0.30049014
  -0.00588423]
 [ 0.5113723   0.1667884  -0.3116603  ...  0.46195886  0.26436093
  -0.00454408]]


In [9]:
max_frames = max(video.shape[0] for video in padded_video_frames)

# Pad the number of frames for each video
X = np.zeros((len(padded_video_frames), 19, max_frames, 1662), dtype='float32')

for i, video in enumerate(padded_video_frames):
    X[i, :video.shape[0]] = video

print(X.shape)

(50, 19, 19, 1662)


In [10]:
y = np.array([label_map[label] for label in labels])
y = to_categorical(y).astype(int)

print(y.shape)

(50, 50)


In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) 

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)



X_train shape: (40, 19, 19, 1662)
X_val shape: (10, 19, 19, 1662)
y_train shape: (40, 50)
y_val shape: (10, 50)


# Model

## Architecture

In [ ]:
tf.keras.backend.clear_session()

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

class INCLUDE50_V1(Model):
    def __init__(self, input_shape, num_classes=50, time_steps=19):
        super().__init__()
        self.num_classes = num_classes
        self.input_shape_ = input_shape
        # self.time_steps = time_steps
        
        # MobileNetV2 base
        # self.MobileNet = MobileNetV2(input_shape=self.input_shape_, include_top=False, weights='imagenet')
        # for layer in self.MobileNet.layers[:-20]:
        #     layer.trainable = False  # Correctly freeze layers
        
        # Apply MobileNetV2 to each time step
        # self.TimeDistribute = TimeDistributed(self.MobileNet)
        self.input_layer = Input(shape=input_shape)
        
        
        # Bidirectional LSTM layers
        self.BD1 = Bidirectional(LSTM(64, return_sequences=True))
        self.BD2 = Bidirectional(LSTM(64, return_sequences=True))
        self.BD3 = Bidirectional(LSTM(64, return_sequences=True))
        self.BD4 = Bidirectional(LSTM(64, return_sequences=True))
        
        # Flatten the output
        self.flatten = Flatten()
        
        # Fully connected layer
        self.dense = Dense(128, activation='relu')
        
        # Output layer
        self.outputs = Dense(self.num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        # x = self.TimeDistribute(inputs)
        
        # # Debugging shape before reshaping
        # print("Shape before Reshape:", x.shape)
        
        # shape = tf.shape(inputs)
        # x = inputs
        # x = Reshape((shape[1], -1))(x)  # Reshape for LSTM
        
        # Debugging shape after reshaping
        # print("Shape after Reshape:", x.shape)
        # print(type(inputs))
        
        # x = self.input_layer(inputs)
        x = self.BD1(inputs)
        x = self.BD2(x)
        x = self.BD3(x)
        x = self.BD4(x)
        
        x = self.flatten(x)
        x = self.dense(x)
        
        return self.outputs(x)

# # Initializing the model and inputing the input shape 


# input_shape = X_train[0].shape #(19, 19, 1662)
# # input_shape = (19, 19, 1662)
# num_classes =  len(label_map.keys())#50
# batch_size = 16


# # input_layer = tf.zeros(input_shape)
# inputs = Input(shape=input_shape)

# model = INCLUDE50_V1(input_shape=input_shape, num_classes=num_classes)
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
# # model.build(input_shape=(None, *input_shape))
# model.build(input_shape=(input_shape))

# model.summary()

In [36]:
import keras

# input_shape = X_train[0].shape #(19, 19, 1662)
input_shape = (19, 19, 1662)
num_classes =  len(label_map.keys())#50
batch_size = 50

# The None Being added here is the betch size so i set it to 40
print(input_shape)
INCLUDESEQ50_V2 = keras.Sequential([
        
        # Input(shape=input_shape),        
        # Input(shape=input_shape),
                
        TimeDistributed(Flatten(), input_shape=input_shape),
        # Bidirectional LSTM layers
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        
        # Flatten the output
        Flatten(),
        
        # Fully connected layer
        Dense(128, activation='relu'),
        
        # Output layer
        Dense(num_classes, activation='softmax')
])

model = INCLUDESEQ50_V2

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
# model.build(input_shape=(None, *input_shape))
model.build(input_shape=(input_shape))

model.summary()

(19, 19, 1662)
Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed (TimeDistr  (None, 19, 31578)        0         
 ibuted)                                                         
                                                                 
 bidirectional_36 (Bidirecti  (None, 19, 128)          16201216  
 onal)                                                           
                                                                 
 bidirectional_37 (Bidirecti  (None, 19, 128)          98816     
 onal)                                                           
                                                                 
 bidirectional_38 (Bidirecti  (None, 19, 128)          98816     
 onal)                                                           
                                                                 
 bidirectional_39 (Bidirecti  (None, 19

In [38]:
model.fit(X_train,
          y_train,
          validation_data=(X_val, y_val), 
          epochs=40,
          batch_size=batch_size)


Epoch 1/40
1/1 [==============================] - 0s 207ms/step - loss: 3.3444 - accuracy: 0.0750 - val_loss: 6.8235 - val_accuracy: 0.0000e+00
Epoch 2/40
1/1 [==============================] - 0s 90ms/step - loss: 3.2333 - accuracy: 0.1250 - val_loss: 7.1565 - val_accuracy: 0.0000e+00
Epoch 3/40
1/1 [==============================] - 0s 92ms/step - loss: 3.1151 - accuracy: 0.1000 - val_loss: 7.5939 - val_accuracy: 0.0000e+00
Epoch 4/40
1/1 [==============================] - 0s 93ms/step - loss: 2.9936 - accuracy: 0.1250 - val_loss: 8.2443 - val_accuracy: 0.0000e+00
Epoch 5/40
1/1 [==============================] - 0s 94ms/step - loss: 2.8756 - accuracy: 0.1500 - val_loss: 9.0487 - val_accuracy: 0.0000e+00
Epoch 6/40
1/1 [==============================] - 0s 95ms/step - loss: 2.7691 - accuracy: 0.1500 - val_loss: 10.0172 - val_accuracy: 0.0000e+00
Epoch 7/40
1/1 [==============================] - 0s 100ms/step - loss: 2.7014 - accuracy: 0.1750 - val_loss: 10.8753 - val_accuracy: 0.0000

In [ ]:
# def downsample_frames(frames, target_height, target_width):
#     """
#     Downsample a batch of frames to the target resolution.

#     Args:
#         frames: A tensor of shape (batch_size, time_steps, height, width, channels).
#         target_height: The desired height for downsampling.
#         target_width: The desired width for downsampling.

#     Returns:
#         A tensor of downsampled frames of shape (batch_size, time_steps, target_height, target_width, channels).
#     """
#     batch_size, time_steps, height, width, channels = frames.shape

#     # Reshape to (batch_size * time_steps, height, width, channels) for resizing each frame
#     reshaped_frames = tf.reshape(frames, (-1, height, width, channels))
    
#     # Resize all frames to the target resolution
#     resized_frames = tf.image.resize(reshaped_frames, [target_height, target_width])
    
#     # Reshape back to the original time-step structure (batch_size, time_steps, target_height, target_width, channels)
#     downsampled_frames = tf.reshape(resized_frames, (batch_size, time_steps, target_height, target_width, channels))
    
#     return downsampled_frames

In [ ]:
# input_shape = ()
# input_shape = (1920, 1080, 3)
# time_steps = 19

# target_height, target_width = 224, 224

# input_layer = tf.zeros((batch_size, time_steps, *input_shape))
# print("Original input shape:", input_layer.shape)

# downsampled_input = downsample_frames(input_layer, target_height, target_width)
# print("Downsampled input shape:", downsampled_input.shape)

# model.build(input_shape=downsampled_input)
